# Cenário: E-commerce (Região Sul)
Este projeto simula um ambiente de E-commerce. Utilizamos duas bases de dados: **Clientes** e **Vendas**. O objetivo é cruzar estas tabelas, filtrar os clientes da região Sul e gravar os dados consolidados.

### Modelo de Entidade-Relacionamento (ER)
O modelo consiste numa relação de **1 para N** (1:N), onde um Cliente pode ter várias Vendas associadas a ele através do `id_cliente`.

![Modelo ER](docs/modelo_er.jpeg)

### Códigos DDL (Estrutura das Tabelas)
Abaixo está a representação da estrutura das nossas tabelas base:

```sql
-- Tabela de Clientes
CREATE TABLE clientes (
    id_cliente INT,
    nome STRING,
    estado STRING
);

-- Tabela de Vendas
CREATE TABLE vendas (
    id_venda INT,
    id_cliente INT,
    data_venda DATE,
    valor FLOAT,
    status STRING
);

# Delta Lake com PySpark
Demonstração de operações CRUD em tabelas Delta Lake usando dados de E-commerce.

In [ ]:
import pyspark
from delta import *
from commons import get_raw_data, filter_southern_region

builder = pyspark.sql.SparkSession.builder.appName("DeltaLab") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print('Spark iniciado com sucesso!')

## Carregando os dados

In [ ]:
df_vendas = get_raw_data(spark, "vendas.csv")
df_clientes = get_raw_data(spark, "clientes.csv")
df_vendas.show(5)
df_clientes.show(5)

## INSERT — Salvando dados no formato Delta Lake

In [ ]:
df_sul = filter_southern_region(df_vendas.join(df_clientes, "id_cliente"))
df_sul.write.format("delta").mode("overwrite").save("../data/delta/vendas_sul")
print('Dados salvos no Delta Lake!')
spark.read.format("delta").load("../data/delta/vendas_sul").show()

## UPDATE — Atualizando valor da venda

In [ ]:
spark.read.format("delta").load("../data/delta/vendas_sul").createOrReplaceTempView("vendas_delta")
spark.sql("UPDATE vendas_delta SET valor = valor * 1.10 WHERE id_venda = 1")
print('UPDATE realizado!')
spark.sql("SELECT * FROM vendas_delta").show()

## DELETE — Removendo uma venda

In [ ]:
spark.sql("DELETE FROM vendas_delta WHERE id_venda = 3")
print('DELETE realizado!')
spark.sql("SELECT * FROM vendas_delta").show()